### PySpark RDD Practice Notes

This notebook is a hands-on reference for learning and practicing the **PySpark RDD API** in Databricks.

---

#### Topics Covered

| # | Topic |
---|-------|
 1 | Explore RDD methods and attributes via `dir()` |
 2 | Word count on hardcoded in-memory data using `flatMap`, `filter`, `map`, `reduceByKey` |
 3 | Partition inspection with `getNumPartitions()` and `glom()` |
 4 | List UC Volume contents using `dbutils.fs.ls()` |
 5 | Read CSV from ADLS Gen2 as RDD using `spark.sparkContext.textFile()` with Hadoop conf key auth |
 6 | Partition count and contents for ADLS-backed RDD |
 7 | Character frequency count on CSV data from ADLS Gen2 |
 8 | Partition inspection and comma frequency count on a second ADLS CSV file |
 9 | Read CSV from ADLS Gen2 as DataFrame using `spark.read.csv()` |

---

#### Key Notes
- `SparkSession` is pre-initialized in Databricks — no need to create it manually.
- ADLS Gen2 (`abfss://`) authentication is done via `spark._jsc.hadoopConfiguration().set()` with a storage account key.
- RDD API does **not** leverage Unity Catalog External Location credential vending; the Hadoop conf must be set explicitly.
- DataFrame API (`spark.read.csv`) works transparently via UC External Location credentials.


---
### SparkSession Initialization (not required in Databricks notebooks)

```
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("WordCountExample").getOrCreate()
```
---

In [0]:
# List available methods and attributes of RDD object
dir(spark.sparkContext.parallelize([]))

In [0]:
# Hardcoded list of words (simulating text input)
data = [
    "hello world this is pyspark",
    "pyspark is great for big data processing",
    "hello spark hello python",
    "data engineering with pyspark and spark"
]
print("Data:- \n", data, end='\n\n')

# Create RDD from hardcoded data
text_rdd = spark.sparkContext.parallelize(data)
# print(text_rdd.collect(), end='\n\n')

# Split lines into words
words = text_rdd.flatMap(lambda line: line.split(" "))
# print(words.collect(), end='\n\n')

# Filter for words containing 'spark'
spark_words = words.filter(lambda w: "spark" in w.lower())
# print(spark_words.collect(), end='\n\n')

# Map each word to (word, 1)
counts = spark_words.map(lambda x: (x, 1))
# print(counts.collect(), end='\n\n')

# Reduce by key to count occurrences of each word containing 'spark'
counts = counts.reduceByKey(lambda a, b: a + b)

# Action: collect and display the word counts for words containing 'spark'
print("Result:- \n", counts.collect())

In [0]:
# Get number of partitions in the RDD
num_partitions = counts.getNumPartitions()
print(num_partitions)

# Show the contents of each partition in the RDD
print(counts.glom().collect())

In [0]:
display(dbutils.fs.ls(r"dbfs:/Volumes/student_acc_dbr_ws/default/managed_volume_1"))

In [0]:
# --- ADLS Gen2 Authentication ---
# Option 1: Using Databricks Secret Scope (recommended — avoids hardcoding keys)
# Note: spark.conf.set for fs.azure.account.key.* is blocked in Unity Catalog SINGLE_USER mode;
# Fetching the secret securely
secret_key = dbutils.secrets.get(scope = "adls_scope", key = "delta0lake0lab0storageac-sa-secret")
# Use it in your Spark configuration
spark.conf.set("fs.azure.account.key.yourstorageaccount.dfs.core.windows.net", secret_key)

# Configure Spark to authenticate to ADLS Gen2 using the account key
# but ABFS reads credentials from the Hadoop conf layer (separate system), so spark.conf.set
spark._jsc.hadoopConfiguration().set(
    "fs.azure.account.key.delta0lake0lab0storageac.dfs.core.windows.net",
    storage_key
)

# Read CSV file as RDD from ADLS Gen2 (abfss:// = Azure Blob File System Secure)
adls_path = 'abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/Spotify_Artists.csv'
csv_rdd = spark.sparkContext.textFile(adls_path)

print("Data:- \n", csv_rdd.take(100))

In [0]:
display(dbutils.secrets.listScopes())

In [0]:
# Get number of partitions in the RDD
num_partitions = csv_rdd.getNumPartitions()
print(num_partitions)

# Show the contents of each partition in the RDD
print(csv_rdd.glom().collect())

In [0]:
# Split lines into words
words = csv_rdd.flatMap(lambda line: line.split(","))
# print(words.collect(), end='\n\n')

# Filter for words containing 'given character'
words_with_char = words.filter(lambda char: "a" in char.lower())
# print(words_with_char.collect(), end='\n\n')

# Map each word to (word, 1)
counts = words_with_char.map(lambda x: (x, 1))
# print(words_with_char.collect(), end='\n\n')

# Reduce by key to count occurrences of each words with 'given character'
counts = counts.reduceByKey(lambda a, b: a + b)

# Action: collect and display the word counts with 'given character'
print("Result:- \n", counts.collect())

In [0]:
# Read CSV file as RDD from ADLS Gen2 (abfss:// = Azure Blob File System Secure)
adls_path = 'abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/Spotify_Listening_Activity.csv'
csv_rdd = spark.sparkContext.textFile(adls_path)
print(csv_rdd.take(10))

# Get number of partitions in the RDD
num_partitions = csv_rdd.getNumPartitions()
print(num_partitions)

# Count occurrences of ',' in the RDD
comma_count = csv_rdd.map(lambda line: line.count(",")).sum()
print("Total number of ',' in the RDD:", comma_count)

In [0]:
# Read CSV file as dataframe from ADLS Gen2
csv_df = spark.read.csv('abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/Spotify_Artists.csv')
print(csv_df.count())